# 05b. 해설위원 목소리 설계 — **Colab GPU**

1. VoiceDesign으로 해설위원 후보 3개 생성 (40~50대 남성, 차분하고 분석적인 톤)
2. 확정된 캐스터(`caster_m30`)와 주고받는 짧은 대화로 어울리는지 들어보기
3. 고른 후보를 Drive `voices/`에 확정 저장 + 목소리별 참조 문장을 `voices.json`에 기록

참조 음성은 Drive `kbo-listen/voices/`에서 읽고 쓴다. (05의 VM 폴더 `tts_test/`는 런타임이 바뀌면 사라짐)

In [ ]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U qwen-tts

In [ ]:
import gc
import json
import shutil
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from google.colab import drive
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

drive.mount("/content/drive")
VOICE_DIR = Path("/content/drive/MyDrive/kbo-listen/voices")
CAND = VOICE_DIR / "analyst_candidates"
CAND.mkdir(parents=True, exist_ok=True)

# 참조 문장 (직접 쓴 일반적인 문장, 실제 중계 원문 아님)
CASTER_REF_TEXT = "안녕하십니까, 오늘도 프로야구 중계로 함께하겠습니다. 양 팀 선발 투수의 맞대결, 시작부터 기대가 되는데요."
ANALYST_REF_TEXT = "네, 그렇습니다. 오늘 선발 투수의 직구 구위가 상당히 좋아 보이는데요, 변화구 제구만 조금 더 잡히면 좋겠습니다."


def load(name):
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")


def free():
    gc.collect()
    torch.cuda.empty_cache()


print(sorted(p.name for p in VOICE_DIR.iterdir()))

In [ ]:
# 1) 해설위원 후보 3개. 이미 만든 후보는 다시 만들지 않는다 (다시 뽑으려면 이름을 바꾸거나 파일을 지움)
CANDIDATES = {  # 이름: (설명, 시드)
    "analyst_a50_calm": ("A Korean male baseball color commentator in his 50s, low and warm voice, calm and analytical, "
                         "relaxed conversational pace, natural falling intonation, never shouting.", 10),
    "analyst_a40_warm": ("A Korean male former baseball player turned TV commentator in his 40s, friendly mid-low voice, "
                         "steady and thoughtful delivery, conversational tone, natural sentence endings.", 11),
    "analyst_a45_clear": ("A Korean male baseball analyst in his mid 40s, clear baritone voice, articulate and composed, "
                          "explains plays calmly at a moderate pace, professional broadcast tone.", 12),
}

design = None
cand_refs = {}
for name, (desc, seed) in CANDIDATES.items():
    path = CAND / f"ref_{name}.wav"
    if path.exists():
        wav, sr = sf.read(path, dtype="float32")
    else:
        if design is None:
            design = load("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
        torch.manual_seed(seed)
        wavs, sr = design.generate_voice_design(text=ANALYST_REF_TEXT, language="Korean", instruct=desc)
        wav = wavs[0]
        sf.write(path, wav, sr)
    cand_refs[name] = (wav, sr)
    print(name)
    display(Audio(wav, rate=sr))

if design is not None:
    del design
    free()

In [ ]:
# 2) 캐스터와 주고받는 대화 샘플: 실제 데이터처럼 Base 모델로 목소리를 복제해서 합성
clone = load("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
caster_ref = sf.read(VOICE_DIR / "ref_caster_m30.wav", dtype="float32")
caster_prompt = clone.create_voice_clone_prompt(ref_audio=caster_ref, ref_text=CASTER_REF_TEXT)

DIALOGUE = [  # (화자, 문장) C=캐스터, A=해설위원
    ("C", "투아웃, 타석에는 김도영."),
    ("C", "바깥쪽 빠졌고요."),
    ("A", "네, 유인구에 잘 참았습니다."),
    ("C", "파울."),
    ("A", "볼 배합이 좋네요. 몸쪽 직구 다음에 바깥쪽 변화구거든요."),
    ("C", "볼카운트 원앤원."),
    ("A", "투 스트라이크 이후에는 아무래도 타자가 불리해지죠."),
    ("C", "쳤습니다! 안타!"),
    ("A", "그렇죠, 노림수가 정확했어요."),
]
GAP = 0.5


def synth(texts, prompt):
    wavs, sr = clone.generate_voice_clone(text=texts, language=["Korean"] * len(texts), voice_clone_prompt=prompt)
    return [np.asarray(w, dtype=np.float32) for w in wavs], sr


caster_lines = [t for s, t in DIALOGUE if s == "C"]
caster_wavs, sr = synth(caster_lines, caster_prompt)

for name, ref in cand_refs.items():
    prompt = clone.create_voice_clone_prompt(ref_audio=ref, ref_text=ANALYST_REF_TEXT)
    analyst_wavs, _ = synth([t for s, t in DIALOGUE if s == "A"], prompt)
    c_iter, a_iter = iter(caster_wavs), iter(analyst_wavs)
    pieces = []
    for speaker, _ in DIALOGUE:  # 대사 순서대로 화자별 음성을 꺼내 이어 붙임
        pieces += [next(c_iter if speaker == "C" else a_iter), np.zeros(int(GAP * sr), np.float32)]
    print(name)
    display(Audio(np.concatenate(pieces), rate=sr))

In [ ]:
# 3) 고른 후보를 확정: voices/에 복사하고, 목소리별 역할·참조 문장을 voices.json에 기록
# a50_calm은 지나치게 힘이 없어서 제외 → 캐스터 2 × 해설위원 2 = 4가지 조합
CHOSEN = ["analyst_a40_warm", "analyst_a45_clear"]
DROPPED = ["analyst_a50_calm"]  # 처음 실행 때 잘못 확정된 것 정리 (후보 폴더의 원본은 남겨둠)

for name in CHOSEN:
    shutil.copy(CAND / f"ref_{name}.wav", VOICE_DIR / f"ref_{name}.wav")
for name in DROPPED:
    (VOICE_DIR / f"ref_{name}.wav").unlink(missing_ok=True)

voices = {
    "caster_m30": {"role": "caster", "ref_text": CASTER_REF_TEXT},
    "caster_m20_v2": {"role": "caster", "ref_text": CASTER_REF_TEXT},
    **{name: {"role": "analyst", "ref_text": ANALYST_REF_TEXT} for name in CHOSEN},
}
(VOICE_DIR / "voices.json").write_text(json.dumps(voices, ensure_ascii=False, indent=1), encoding="utf-8")
print(sorted(p.name for p in VOICE_DIR.iterdir()))
print(json.dumps(voices, ensure_ascii=False, indent=1))